<a href="https://colab.research.google.com/github/Preetitamrakar-phd/GenAI_Hands-on/blob/main/Agentic_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Agentic RAG - LangChain Agent with Tools**

1. Creating custom tools for RAG retrieval
2. Building an agent with tool calling
3. Handling conversational context
4. Multi-step reasoning and tool selection
5. Comparing agent-based vs direct RAG

In [1]:
import os
import json
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage, SystemMessage

from tools import SupportTicketTools

In [2]:
# Load environment variables
load_dotenv()

True

In [4]:
print("AGENTIC RAG: LangChain Agent with RAG Tools")

print("\nThis project shows how to build an intelligent agent that:")
print("✓ Uses RAG retrieval as a tool (not the only approach)")
print("✓ Decides when to use which tool based on the query")
print("✓ Maintains conversation context across turns")
print("✓ Performs multi-step reasoning")
print("✓ Shows decision trace for tool selection")

AGENTIC RAG: LangChain Agent with RAG Tools

This project shows how to build an intelligent agent that:
✓ Uses RAG retrieval as a tool (not the only approach)
✓ Decides when to use which tool based on the query
✓ Maintains conversation context across turns
✓ Performs multi-step reasoning
✓ Shows decision trace for tool selection


PART 1: Setup Agent with Tools

In [5]:
print("\nInitializing LLM...")
llm = ChatOpenAI(
    model=os.getenv('OPENAI_CHAT_MODEL', 'gpt-4o-mini'),
    temperature=0,
    api_key=os.getenv('OPENAI_API_KEY')
)
print("✓ LLM initialized")


Initializing LLM...
✓ LLM initialized


In [13]:
print("\nCreating agent tools...")
tool_manager = SupportTicketTools(tickets_path="synthetic_tickets.json")
tools = tool_manager.get_tools()


Creating agent tools...


In [14]:
# Convert tools to OpenAI function format
tool_definitions = []
for tool in tools:
    tool_definitions.append({
        "type": "function",
        "function": {
            "name": tool.name,
            "description": tool.description,
            "parameters": {
                "type": "object",
                "properties": {
                    "input": {
                        "type": "string",
                        "description": "The input to the tool"
                    }
                },
                "required": ["input"]
            }
        }
    })


In [15]:
# Bind tools to LLM
llm_with_tools = llm.bind(tools=tool_definitions)

In [16]:
print(f"✓ Created {len(tools)} tools:")
for tool in tools:
    print(f"  • {tool.name}: {tool.description.split('.')[0]}")

✓ Created 4 tools:
  • SearchSimilarTickets: Use this tool to search for similar support tickets based on a problem description or question
  • GetTicketByID: Use this tool to retrieve details of a specific ticket by its ID
  • SearchByCategory: Use this tool to find all tickets in a specific category
  • GetTicketStatistics: Use this tool to get statistics about the ticket database including total count, 
category distribution, and priority breakdown


In [17]:
def run_agent(query: str, max_iterations: int = 5) -> str:
    """
    Run a ReAct-style tool-calling loop until the model returns a final answer.

    Loop behavior:
    1) Model sees conversation + tool schema.
    2) Model either answers directly OR emits one/more tool calls.
    3) We execute each tool call, append ToolMessage results.
    4) Repeat until no tool calls remain or iteration cap is reached.
    """
    messages = [
        SystemMessage(content="""You are an expert support desk assistant that helps troubleshoot technical issues.

You have access to a database of previous support tickets with their resolutions.
Use your tools to find relevant information and provide helpful, accurate answers.

Guidelines:
- ALWAYS search for similar tickets when asked about troubleshooting or "how to fix" questions
- Be specific and reference ticket IDs when providing solutions
- If multiple similar issues exist, mention the most relevant ones
- Admit when you don't have enough information
- Be concise but thorough in your responses
- When appropriate, use multiple tools to gather complete information
- Before each tool call, provide a short public rationale in content using this exact prefix:
    "Decision: <one sentence explaining why this tool is needed>"

Remember: Your primary value is retrieving and applying solutions from past tickets!"""),
        HumanMessage(content=query)
    ]

    for i in range(max_iterations):
        response = llm_with_tools.invoke(messages)
        messages.append(response)

        # If the model produced no tool calls, treat content as final answer.
        if not response.tool_calls:
            # No more tool calls, return the response
            return response.content

        # Print model-provided public rationale (not hidden chain-of-thought).
        decision_trace = (response.content or "").strip()
        if decision_trace:
            print(f"\n🧭 {decision_trace}")

        # Execute each requested tool exactly as the model specified.
        for tool_call in response.tool_calls:
            tool_name = tool_call["name"]
            tool_input = tool_call["args"].get("input", "")

            print(f"\n🔧 Calling tool: {tool_name}")
            print(f"   Input: {tool_input}")

            # Resolve tool by name from the registered tool list.
            # This explicit lookup keeps control in application code (safer than eval).
            tool_output = None
            for tool in tools:
                if tool.name == tool_name:
                    tool_output = tool.func(tool_input)
                    break

            if tool_output is None:
                tool_output = f"Error: Tool {tool_name} not found"

            print(f"   Output: {tool_output[:200]}...")

            # Feed tool output back to model in the expected ToolMessage format.
            # The `tool_call_id` links this output to the originating request.
            messages.append(ToolMessage(
                content=tool_output,
                tool_call_id=tool_call["id"]
            ))

    return "Maximum iterations reached. Could not complete the task."

print("\n✓ Agent ready!")


✓ Agent ready!


PART 2: Simple Query - RAG Tool Selection

In [23]:
query1 = "How do I fix authentication problems after password reset?"
print(f"\nQuery: '{query1}'")

response1 = run_agent(query1)
print("\n" + "-" * 80)
print("FINAL ANSWER:\n")
print(response1)


Query: 'How do I fix authentication problems after password reset?'

🔧 Calling tool: SearchSimilarTickets
   Input: authentication problems after password reset
   Output: Found similar tickets:

--- Ticket 1 ---
Ticket ID: TICK-001
Title: Users unable to log in after password reset
Description: Multiple users reporting authentication failures after performing password ...

--------------------------------------------------------------------------------
FINAL ANSWER:

To address authentication problems after a password reset, you can refer to the resolution from a similar ticket (TICK-001). Here’s what was done to resolve the issue:

1. **Issue Identified**: Users were unable to log in after performing a password reset, receiving an "Invalid credentials" error. This was linked to a recent update in the password hash algorithm without invalidating session tokens.

2. **Solution Implemented**:
   - **Clear Active Sessions**: All active sessions were cleared to ensure that users would ne

PART 3: Specific Lookup - Different Tool

In [22]:
print("PART 3: Specific Ticket Lookup")

query2 = "Show me details of ticket TICK-005"
print(f"\nQuery: '{query2}'")

response2 = run_agent(query2)
print("\n" + "-" * 80)
print("FINAL ANSWER:")
print(response2)

PART 3: Specific Ticket Lookup

Query: 'Show me details of ticket TICK-005'

🔧 Calling tool: GetTicketByID
   Input: TICK-005
   Output: Ticket ID: TICK-005
Title: Memory leak in background worker process
Description: Background job processor showing steady memory increase over time. Process killed by OS after reaching 8GB memory usage...

--------------------------------------------------------------------------------
FINAL ANSWER:
Here are the details for ticket **TICK-005**:

- **Title**: Memory leak in background worker process
- **Description**: The background job processor was showing a steady increase in memory usage over time, leading to the process being killed by the OS after reaching 8GB. A daily restart was required to maintain functionality.
- **Resolution**: Memory profiling revealed unclosed database cursors in the batch processing loop. The solution involved adding explicit `cursor.close()` calls and implementing context managers for all database operations. After these

PART 4: Category Filtering

In [24]:
query3 = "What payment-related issues have we seen?"
print(f"\nQuery: '{query3}'")

response3 = run_agent(query3)
print("\n" + "-" * 80)
print("FINAL ANSWER:")
print(response3)


Query: 'What payment-related issues have we seen?'

🔧 Calling tool: SearchByCategory
   Input: Payment
   Output: Found 1 tickets in category 'Payment':

• [TICK-003] Payment processing fails for international cards (Priority: High)
...

--------------------------------------------------------------------------------
FINAL ANSWER:
We have one payment-related issue recorded:

- **Ticket ID:** TICK-003
  - **Issue:** Payment processing fails for international cards
  - **Priority:** High

If you need more details about this ticket or its resolution, please let me know!


PART 5: Statistics Query

In [25]:
query4 = "Give me an overview of the ticket database"
print(f"\nQuery: '{query4}'")

response4 = run_agent(query4)
print("\n" + "-" * 80)
print("FINAL ANSWER:")
print(response4)


Query: 'Give me an overview of the ticket database'

🔧 Calling tool: GetTicketStatistics
   Input: 
   Output: Ticket Database Statistics:
Total Tickets: 20

By Category:
  • Authentication: 4
  • Performance: 2
  • API: 2
  • Database: 1
  • Payment: 1
  • Mobile: 1
  • Email: 1
  • Search: 1
  • File Upload:...

--------------------------------------------------------------------------------
FINAL ANSWER:
Here is an overview of the ticket database:

- **Total Tickets**: 20

**By Category**:
- Authentication: 4
- Performance: 2
- API: 2
- Database: 1
- Payment: 1
- Mobile: 1
- Email: 1
- Search: 1
- File Upload: 1
- Reporting: 1
- Integration: 1
- Export: 1
- Real-time: 1
- Media Processing: 1
- User Management: 1

**By Priority**:
- High: 9
- Medium: 6
- Critical: 5

If you need more specific information or details about any category or ticket, feel free to ask!


PART 6: Multi-Step Reasoning

In [26]:
query5 = "Find database-related critical issues and tell me how they were resolved"
print(f"\nQuery: '{query5}'")

response5 = run_agent(query5)
print("\n" + "-" * 80)
print("FINAL ANSWER:")
print(response5)


Query: 'Find database-related critical issues and tell me how they were resolved'

🔧 Calling tool: SearchByCategory
   Input: Database
   Output: Found 1 tickets in category 'Database':

• [TICK-002] Database connection timeout in production (Priority: Critical)
...

🔧 Calling tool: GetTicketByID
   Input: TICK-002
   Output: Ticket ID: TICK-002
Title: Database connection timeout in production
Description: Application experiencing intermittent 500 errors. Logs show 'connection pool exhausted' and 'timeout waiting for conne...

--------------------------------------------------------------------------------
FINAL ANSWER:
The critical database issue identified was related to a "Database connection timeout in production" (Ticket ID: TICK-002). 

### Resolution:
- The database connection pool was initially sized too small for the peak load, leading to intermittent 500 errors.
- The following actions were taken to resolve the issue:
  - Increased the `max_connections` from 20 to 100.
  - A

PART 7: Conversational Agent with Memory

In [27]:
print("\nSimulating a multi-turn conversation...")


Simulating a multi-turn conversation...


In [28]:
def run_conversational_agent(conversation_history, query: str, max_iterations: int = 5) -> tuple:
    """
    Run the agent while preserving prior conversation turns.

    `conversation_history` should already contain Human/AI/Tool messages from
    previous turns so follow-up questions can resolve references like "that".
    """
    messages = [SystemMessage(content="""You are an expert support desk assistant that helps troubleshoot technical issues.
Use your tools to find relevant information and maintain context across our conversation.
Before each tool call, provide a short public rationale in content using this exact prefix:
"Decision: <one sentence explaining why this tool is needed>".""")]

    # Replay prior turns before appending the new user query.
    messages.extend(conversation_history)
    messages.append(HumanMessage(content=query))

    for i in range(max_iterations):
        response = llm_with_tools.invoke(messages)
        messages.append(response)

        if not response.tool_calls:
            return messages, response.content

        decision_trace = (response.content or "").strip()
        if decision_trace:
            print(f"\n🧭 {decision_trace}")

        for tool_call in response.tool_calls:
            tool_name = tool_call["name"]
            tool_input = tool_call["args"].get("input", "")

            print(f"\n🔧 Calling tool: {tool_name}")

            tool_output = None
            for tool in tools:
                if tool.name == tool_name:
                    tool_output = tool.func(tool_input)
                    break

            if tool_output is None:
                tool_output = f"Error: Tool {tool_name} not found"

            messages.append(ToolMessage(
                content=tool_output,
                tool_call_id=tool_call["id"]
            ))

    return messages, "Maximum iterations reached."

In [29]:
# Start conversation
conversation = []

print("\n--- Conversation Turn 1 ---")
conv_query1 = "What issues have we had with iOS?"
print(f"User: {conv_query1}\n")
conversation, conv_response1 = run_conversational_agent(conversation, conv_query1)
print(f"\nAssistant: {conv_response1}")


--- Conversation Turn 1 ---
User: What issues have we had with iOS?


🔧 Calling tool: SearchByCategory

🔧 Calling tool: SearchByCategory

Assistant: We have had one issue related to iOS, specifically in the Mobile category:

- **Ticket ID:** TICK-004
- **Description:** Mobile app crashes on iOS 17
- **Priority:** Critical

If you need more details about this ticket, please let me know!


In [30]:
print("\n--- Conversation Turn 2 (Follow-up) ---")
conv_query2 = "What was the ticket ID for that?"
print(f"User: {conv_query2}")
print("(Notice: The agent remembers the previous context!)\n")
conversation, conv_response2 = run_conversational_agent(conversation, conv_query2)
print(f"\nAssistant: {conv_response2}")


--- Conversation Turn 2 (Follow-up) ---
User: What was the ticket ID for that?
(Notice: The agent remembers the previous context!)


Assistant: The ticket ID for the issue related to the mobile app crashing on iOS 17 is **TICK-004**. If you need further information about this ticket, just let me know!


In [31]:
print("\n--- Conversation Turn 3 (Another Follow-up) ---")
conv_query3 = "How was it resolved?"
print(f"User: {conv_query3}\n")
conversation, conv_response3 = run_conversational_agent(conversation, conv_query3)
print(f"\nAssistant: {conv_response3}")


--- Conversation Turn 3 (Another Follow-up) ---
User: How was it resolved?


🔧 Calling tool: GetTicketByID

Assistant: The issue with the mobile app crashing on iOS 17 (Ticket ID: TICK-004) was resolved as follows:

- **Description:** The app crashed immediately on launch for users who upgraded to iOS 17, with crash reports indicating an EXC_BAD_ACCESS error in the networking module. This affected approximately 30% of the iOS user base.
- **Resolution:** The problem was due to iOS 17 deprecating certain networking APIs that were being used. The team updated the URLSession implementation to use the new async/await pattern and removed the deprecated NSURLConnection usage. An updated build was then submitted to the App Store to ensure compatibility with iOS 17.
- **Created:** January 22, 2024
- **Resolved:** January 23, 2024

If you have any more questions or need further assistance, feel free to ask!
